# 03 · Hypothesis Testing & Confidence Intervals

Lecture 1 — From Messy Medical Data to Statistical Insight

*Created by Master of AI.*

> **Realistic training data, prepared for teaching.**

In [ ]:
import numpy as np
import pandas as pd
pd.set_option("display.width", 160)
DATA = "../data/"
from scipy import stats
from scipy.stats import ttest_ind
df = pd.read_csv(DATA + "clean_patient_screening_data.csv", parse_dates=["visit_date"])
print(df.shape)
df.head()

## 1. Question
Is mean fasting glucose different between smokers and non-smokers?

- H0: μ(smoker) = μ(non-smoker)
- H1: μ(smoker) ≠ μ(non-smoker)
- α = 0.05

In [ ]:
group_a = df.loc[df["smoking_status"] == "Non-Smoker", "fasting_glucose"].dropna()
group_b = df.loc[df["smoking_status"] == "Smoker", "fasting_glucose"].dropna()
print(len(group_a), len(group_b), round(group_a.mean(), 2), round(group_b.mean(), 2))

## 2. Check assumptions
Skewed data and unequal variances -> Welch's t-test, plus a rank-based check.

In [ ]:
print("skew:", round(group_a.skew(), 2), round(group_b.skew(), 2))
print("Levene p:", stats.levene(group_a, group_b).pvalue)

In [ ]:
stat, p_value = ttest_ind(group_a, group_b, equal_var=False)
print("t-statistic:", stat)
print("p-value:", p_value)
print("Mann-Whitney U p:", stats.mannwhitneyu(group_a, group_b).pvalue)

**Interpretation.** The p-value is the probability of observing a difference at least this large *if H0 were true*. It is **not** the probability that H0 is true. A large p-value means *insufficient evidence*, not *proof of no difference*.

## 3. Effect size — practical significance

In [ ]:
na, nb = len(group_a), len(group_b)
pooled = np.sqrt(((na-1)*group_a.var() + (nb-1)*group_b.var()) / (na+nb-2))
print("Cohen d:", round((group_b.mean() - group_a.mean()) / pooled, 3))

## 4. 95% confidence interval for a mean

In [ ]:
g = df["fasting_glucose"].dropna()
se = g.std(ddof=1) / np.sqrt(len(g))
t_crit = stats.t.ppf(0.975, len(g) - 1)
print(f"mean {g.mean():.2f}, 95% CI [{g.mean()-t_crit*se:.2f}, {g.mean()+t_crit*se:.2f}]")

**Correct interpretation:** if we repeated the sampling many times and built an interval each time, about 95% of those intervals would contain the true mean. We do **not** say there is a 95% probability that the true value lies in this particular interval.

## 5. Simulation — coverage of repeated 95% intervals (systolic BP, n = 50)

In [ ]:
rng = np.random.default_rng(42)
pop = df["systolic_bp"].dropna().values; mu = pop.mean(); hits = 0
for _ in range(1000):
    s = rng.choice(pop, 50, replace=False)
    half = stats.t.ppf(.975, 49) * s.std(ddof=1) / np.sqrt(50)
    hits += abs(s.mean() - mu) <= half
print("coverage %:", hits / 10)